# Suite REF — References: one format for objects, relation entries and frames

A reference designates what a request or a response points at ([References](../../docs/proposals/references/README.md);
FRAMEWORK.md, Resolved, References): `References.OfObject`, an object, by its path (a singleton, an owned object or a
schema object) or, a counted object an upsert made, by what made it, the frame that reached the upsert, its key and the
symbol (docs/OBJECT_IDENTITY.md, Decided, 61, 63 and 102); `References.OfEntry`, a relation entry, by its relation's
path and all its links and properties; `References.OfFrame`, a frame, each symbol's object. `References.of(store, x)`
gives one, a value of the store's, written and read as any value is; `References.resolve(store, reference)` gives what
it designates, or None where the store holds none.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
from mbse.Schemas.Framework import JSON, Builders as B, Paths, Plain, References
from support import raises


def crm(txn):
    """crm.Person, who knows persons since a year; crm.Book, a singleton owning its contacts by key; crm.Note, counted,
    about a person."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Knows": S.OfRelation, "Person": S.OfObject, "Holds": S.OfRelation, "Book": S.OfObject,
                  "Note": S.OfObject, "About": S.OfRelation})
        .spec([
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")]).properties({
                "since": S.OfInt}),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr}).adjacencies([
                lambda t: t.name("knows").relation([_.Knows]).me("knower"),
                lambda t: t.relation([_.Knows]).me("known"),
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True),
            ]).adjacencies({"trio": lambda t: t.relation([S.OfRelation(lambda r: r.links([  # no name of its own
                lambda t: t.name("a"), lambda t: t.name("b"), lambda t: t.name("c")]))]).me("a")}),
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")]).properties({
                "key": S.OfStr}).uniques([["book", "key"], ["contact"]]),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book")}),
            lambda _: _.Note.name("crm.Note").ref(True).properties({"text": S.OfStr}).adjacencies({
                "about": lambda t: t.relation([_.About]).me("note")}),
            lambda _: _.About.name("crm.About").links([lambda t: t.name("note"), lambda t: t.name("person")]),
        ])]))])


store = B.Store()
store.transaction(crm)
Person, Book, Note = (store.registered(name) for name in ("crm.Person", "crm.Book", "crm.Note"))
book = store.singleton("crm.TheBook")
frames = store.transaction(lambda t: t.uid("people").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
    .symbols({"book": Book, "ada": Person, "bob": Person, "cy": Person}).spec([
        lambda _: _.book.contacts({"ada": lambda e: e.contact(_.ada), "bob": lambda e: e.contact(_.bob),
                                   "cy": lambda e: e.contact(_.cy)}),
        lambda _: _.ada.name("Ada").knows([lambda e: e.known(_.bob).since(2020), lambda e: e.known(_.cy).since(2021)]),
        lambda _: _.bob.name("Bob"),
        lambda _: _.cy.name("Cy"),
    ])]))]))
ada, bob, cy = (frames[0][name] for name in ("ada", "bob", "cy"))


def notes(t):
    """A note about each person, upserted with the key `notes/1` for the frame that reaches it."""
    return t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols({"contact": Person})])
        .then([lambda n: n.upsert(lambda u: u
        .key("notes/1").exists([lambda p: p.symbols({"contact": Person, "note": Note}).spec([
            lambda _: _.note.text("hi").about([_.contact])])]))]))])


def plain(reference):
    return Plain.ToPlain(store)(reference.as_schema(), reference)


def path(singleton, *steps):
    """A path's plain form: from a singleton through entries, each (i, o, keys)."""
    nested = {"target": {}}
    for i, o, keys in reversed(steps):
        nested = {"entry": {"i": i, "o": o, "properties": keys, "next": nested}}
    return {"singleton": singleton, "next": nested}

## REF-01 · Every store holds the reference schemas, its own; References.install, run again, writes nothing

In [ ]:
OfObject, OfUpserted = store.registered("References.OfObject"), store.registered("References.OfUpserted")
OfFrame, OfEntry = store.registered("References.OfFrame"), store.registered("References.OfEntry")
assert [(branch.name, branch.schema) for branch in OfObject.branches] == [
    ("path", store.registered("Paths.OfRoot")), ("upserted", OfUpserted)]
assert list(OfUpserted.properties) == ["frame", "key", "symbol"]
assert OfUpserted.properties["frame"].schema is OfFrame and OfFrame.properties["binds"].schema.element is OfObject
assert list(OfEntry.properties) == ["relation", "links", "properties"]
assert OfEntry.properties["relation"].schema is store.registered("Paths.OfRoot")
assert OfEntry.properties["links"].schema.element is OfObject
assert not any(schema.ref for schema in (OfUpserted, OfFrame, OfEntry))  # value objects, as a path is
other = B.Store()
assert other.registered("References.OfObject") is not OfObject  # each store's its own
names = {name: store.registered(name) for name in store.names()}
store.transaction(References.install)  # each found by its name
assert {name: store.registered(name) for name in store.names()} == names

## REF-02 · An object's reference by its path: a singleton, an owned object, a schema object; each resolved

In [ ]:
assert plain(References.of(store, book)) == {"path": path("crm.TheBook")}
assert plain(References.of(store, ada)) == {"path": path("crm.TheBook", ("contacts", "book", {"key": "ada"}))}
assert plain(References.of(store, Person)) == {"path": path(".", ("schemas", "", {"name": "crm"}),
                                                              ("schemas", "", {"name": "Person"}))}
for target in (book, ada, Person):
    assert References.resolve(store, References.of(store, target)) is target
built = store.References.OfObject(lambda o: o.path(lambda p: p.singleton("crm.TheBook").next(lambda n: n.entry(
    lambda e: e.i("contacts").o("book").properties({"key": "bob"}).next(lambda n: n.target(lambda t: t))))))
assert References.resolve(store, built) is bob  # written through the builders' fluent calls

## REF-03 · A counted object an upsert made: by its frame, key and symbol; found again when the transaction runs again

In [ ]:
made = store.transaction(notes)
assert [frame["contact"] for frame in made] == [ada, bob, cy]
note = made[0]["note"]
reference = References.of(store, note)
assert plain(reference) == {"upserted": {
    "frame": {"binds": {"contact": {"path": path("crm.TheBook", ("contacts", "book", {"key": "ada"}))}}},
    "key": "notes/1", "symbol": "note"}}
assert References.resolve(store, reference) is note
again = store.transaction(notes)  # run again: found by what made it, not made again
assert again[0]["note"] is note and plain(References.of(store, again[0]["note"])) == plain(reference)
assert References.resolve(store, reference) is note
assert References.resolve(store, References.of(store, made[1]["note"])) is made[1]["note"] is not note
keyless = store.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
    {"note": Note}).spec([lambda _: _.note.text("bare")])]))]))[0]["note"]  # an upsert with no key, an empty frame
assert plain(References.of(store, keyless)) == {"upserted": {"frame": {"binds": {}}, "symbol": "note"}}
assert References.resolve(store, References.of(store, keyless)) is keyless
built = store.References.OfObject(lambda o: o.upserted(lambda u: u.symbol("note")))  # no frame: the empty one
assert References.resolve(store, built) is keyless

## REF-04 · A relation entry by its relation's path and all its links and properties: a keyed relation's, one without properties

In [ ]:
held = next(iter(book.contacts))  # crm.Holds, keyed by `key`
assert plain(References.of(store, held)) == {
    "relation": path(".", ("schemas", "", {"name": "crm"}), ("schemas", "", {"name": "Holds"})),
    "links": {"book": {"path": path("crm.TheBook")},
              "contact": {"path": path("crm.TheBook", ("contacts", "book", {"key": "ada"}))}},
    "properties": {"key": {"str": "ada"}}}
knows = list(ada.knows)
assert plain(References.of(store, knows[1]))["properties"] == {"since": {"int": 2021}}
about = next(iter(note.about))  # crm.About: no properties
assert plain(References.of(store, about))["properties"] == {}
for entry in (held, knows[0], knows[1], about):
    assert References.resolve(store, References.of(store, entry)) is entry
changed = store.References.OfEntry(lambda e: e.relation(Paths.of(store.registered("crm.Knows"))).links({
    "knower": lambda o: o.path(Paths.of(ada)), "known": lambda o: o.path(Paths.of(cy))}).properties({"since": 1999}))
assert References.resolve(store, changed) is None  # a property no entry has
fewer = store.References.OfEntry(lambda e: e.relation(Paths.of(store.registered("crm.Knows"))).links({
    "knower": lambda o: o.path(Paths.of(ada))}))
assert References.resolve(store, fewer) is None  # every link, or none
assert References.resolve(store, store.References.OfEntry(lambda e: e.relation(Paths.of(store.registered("crm.Knows"))))) \
    is None  # no links

## REF-05 · A frame a transaction gave back: each symbol's object, resolved to the frame

In [ ]:
frame = made[2]
reference = References.of(store, frame)
assert sorted(plain(reference)["binds"]) == ["contact", "note"]
assert plain(reference)["binds"]["note"] == plain(References.of(store, frame["note"]))
assert References.resolve(store, reference) == frame
assert plain(References.of(store, {})) == {"binds": {}} and References.resolve(store, References.of(store, {})) == {}
assert References.resolve(store, store.References.OfFrame(lambda f: f)) == {}  # no binds: the empty frame

## REF-06 · What is refused: a counted object nothing names, another store's object, what is not an object, an entry or a frame

In [ ]:
with store.transaction():
    loose = store.crm.Note(lambda n: n.text("by hand"))  # counted, written outright: nothing names it
with raises(LookupError, match="a counted 'crm.Note' that no upsert made has no reference: nothing names it"):
    References.of(store, loose)
with raises(LookupError, match="a counted 'crm.Note' that no upsert made has no reference"):
    References.of(store, {"loose": loose})
with raises(TypeError, match="the object belongs to another store"):
    References.of(other, ada)
with raises(TypeError, match="References.of takes an object, a relation entry or a frame, not 'x'"):
    References.of(store, "x")
with store.transaction():
    store.crm.Person(ada, lambda a: a.trio(lambda t: t.insert(lambda e: e.b(bob).c(cy))))
with raises(LookupError, match="has no path"):  # an entry of a relation with no name of its own
    References.of(store, next(iter(ada.trio)))
for wrong in ("x", Paths.of(ada)):
    with raises(TypeError, match="References.resolve takes a References.OfObject, OfEntry or OfFrame value of the store's"):
        References.resolve(store, wrong)

## REF-07 · A reference written and read back as a value; None where the store holds nothing it designates

In [ ]:
held = next(iter(book.contacts))  # as the store holds it now: an update of ada's rewrote her entries
for target in (ada, note, held, frame):
    reference = References.of(store, target)
    text = JSON.ToJSON(store)(reference.as_schema(), reference)
    back = JSON.FromJSON(store)(reference.as_schema(), text)
    found = References.resolve(store, back)
    assert found == target if isinstance(target, dict) else found is target
empty = B.Store()  # its schemas and nothing else
empty.transaction(crm)
for target in (ada, note, held, frame, Person):
    reference = References.of(store, target)
    back = JSON.FromJSON(empty)(empty.registered(store.name_of(reference.as_schema())),
                                JSON.ToJSON(store)(reference.as_schema(), reference))
    assert References.resolve(empty, back) is (empty.registered("crm.Person") if target is Person else None)
bare = B.Store()  # not even the schemas
reference = References.of(store, held)
assert References.resolve(bare, JSON.FromJSON(bare)(bare.References.OfEntry.as_schema(),
                                                    JSON.ToJSON(store)(reference.as_schema(), reference))) is None
unheld = store.References.OfObject(lambda o: o.upserted(lambda u: u.symbol("nothing")))  # no upsert made it
nowhere = store.References.OfObject(lambda o: o.path(lambda p: p.singleton(".").next(lambda n: n.entry(
    lambda e: e.i("schemas").o("x").properties({"name": "crm"}).next(lambda n: n.target(lambda t: t))))))
gone = store.References.OfEntry(lambda e: e.relation(Paths.of(Person)).links({"knower": lambda o: o.path(Paths.of(ada))}))
assert References.resolve(store, unheld) is References.resolve(store, nowhere) is References.resolve(store, gone) is None